### data ingetion


In [1]:
from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is where content resides q",
    metadata={
        "source":"example.txt",
        "pages":"1",
        "author":"Ishan",
        "date_created":"2026-8-7"
    }
)

In [3]:

sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [4]:
##text loader
from langchain_community.document_loaders  import TextLoader

/home/ishan/ISHAN/CODING/coding/RAG/ytrag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
loader=TextLoader("../data/text_files/python_intro.txt")
dox=loader.load()
print(dox)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
# #dirctory loadeer
# from langchain_community.document_loaders import DirectoryLoader
# dir_loader=DirectoryLoader(
#     "../data/text_files",
#     glob="**/*.txt",
#     loader_cls=TextLoader,
#     loader_kwargs={"encoding":'utf-8'},
#     show_progress=True
# )

# doc=dir_loader.load()
# print(doc)

In [7]:
from langchain_community.document_loaders import PyPDFLoader
pdf_loader=PyPDFLoader('../data/pdf_files/Policy-Document.pdf')
doc=pdf_loader.load()
doc

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}, page_content="DOCUMENT\nPOLICY\nST. PAUL ’S COLLEGE, Kalamassery\nRe-accredited with 'A' Grade (Third Cycle) by NAAC\n(Aﬃliated to Mahatma Gandhi University, Kottayam)\nKalamassery, HMT P .O., Ernakulam- 683 503. Kerala-India"),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 1, 'page_label': '2'}, page_content='1 | ST. PAUL’S COLLEGE, KALAMASSERY \n \nPolicy Document'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 2, 'page_label': '3'}, page_content='2 | ST. 

In [8]:
import numpy as np
import chromadb
import uuid
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [9]:
class EmbeddingManager:
    """handles document embedding generation using sentenceTransformer"""
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """load the sentenseTransformer model"""
        try:
            print("loading the model:",self.model_name)
            self.model=SentenceTransformer(self.model_name)
            print("model loaded with embedding : ",self.model.get_embedding_dimension())
        except Exception as e:
            print("error loading the model",e)
            raise
    def generate_embedding(self,text:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("model not init")
        print(f"generateing embedding for {len(text)} texts")
        embedings=self.model.encode(text,show_progress_bar=True)
        print(f"embeding generated with shape {embedings.shape}")
        return embedings
##initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


loading the model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1896.37it/s]


model loaded with embedding :  384


### vector store


In [10]:
import os

In [11]:

class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persisit_dir:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_dir=persisit_dir
        self.client=None
        self.collection=None
        self.embedding_matrix = None
        self.document_texts: List[str] = []
        self.doc_ids: List[str] = []
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_dir)
            self.collection=self.client.get_or_create_collection(self.collection_name,metadata={"description":"PDF embeddings for RAG"})
            print(f"collection initialized with name {self.collection_name}")
            print(f"document count in collection {self.collection.count()}")
        except Exception as e:
            print("error creating db collection {e}")
            raise
    def add_document(self,documents:List[Any],embedding:np.ndarray):
        if len(documents)!=len(embedding):
            raise ValueError("number of documet must be same")
        print(f"adding {len(documents)} to vector database")
        ids=[]
        metadatas=[]
        document_text=[]
        embedding_list=[]
        for i,(doc,embeds)in enumerate(zip(documents,embedding)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['document_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content)
            embedding_list.append(embeds.tolist())
        try:
            self.collection.add(ids,embedding_list,metadatas,documents=document_text)
            self.embedding_matrix = np.asarray(embedding_list)
            self.document_texts = document_text
            self.doc_ids = ids
            print(f"succefully added {len(documents)} documents to vector store")
            print(f"total documents in collection {self.collection.count()}")
        except Exception as e:
            print("error adding into vector store : ",e)
            raise
    def get_all_embeddings(self):
        if self.embedding_matrix is None:
            raise ValueError("No embeddings loaded in memory")
        return self.embedding_matrix

    def get_all_documents(self):
        return self.doc_ids, self.document_texts, self.embedding_matrix
vectorstore=VectorStore()
vectorstore

collection initialized with name pdf_documents
document count in collection 32


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=100,chunk_overlap=20):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [13]:
chunks=split_documents(doc)
chunks

Split 50 documents into 958 chunks

Example chunk:
Content: DOCUMENT
POLICY
ST. PAUL ’S COLLEGE, Kalamassery
Re-accredited with 'A' Grade (Third Cycle) by NAAC...
Metadata: {'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}


[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}, page_content="DOCUMENT\nPOLICY\nST. PAUL ’S COLLEGE, Kalamassery\nRe-accredited with 'A' Grade (Third Cycle) by NAAC"),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}, page_content='(Aﬃliated to Mahatma Gandhi University, Kottayam)'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}, page_content='Kalamassery, HMT P .O., Ernakulam- 683 503. Kerala-India'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 

In [14]:
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}, page_content="DOCUMENT\nPOLICY\nST. PAUL ’S COLLEGE, Kalamassery\nRe-accredited with 'A' Grade (Third Cycle) by NAAC"),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}, page_content='(Aﬃliated to Mahatma Gandhi University, Kottayam)'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-01-12T04:56:11+00:00', 'source': '../data/pdf_files/Policy-Document.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}, page_content='Kalamassery, HMT P .O., Ernakulam- 683 503. Kerala-India'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 

In [15]:
texts=[docu.page_content for docu in chunks]
texts

["DOCUMENT\nPOLICY\nST. PAUL ’S COLLEGE, Kalamassery\nRe-accredited with 'A' Grade (Third Cycle) by NAAC",
 '(Aﬃliated to Mahatma Gandhi University, Kottayam)',
 'Kalamassery, HMT P .O., Ernakulam- 683 503. Kerala-India',
 '1 | ST. PAUL’S COLLEGE, KALAMASSERY \n \nPolicy Document',
 '2 | ST. PAUL’S COLLEGE, KALAMASSERY \n \nPolicy Document',
 '3 | ST. PAUL’S COLLEGE, KALAMASSERY \n \nPolicy Document  \n \nCode of Conduct   \n   \n Introduction',
 'Introduction \n \nA code of conduct describes the values, vision, mission, and principles of the',
 'college. The code acts as a standard that the staff need to meet and informs them',
 'about the best practices. It is the  responsibility of the staff to be aware of the',
 'standards set out in this code and to always apply these standards. The infringement',
 'of this code may lead to disciplinary action. All employees shall respect and adhere',
 'to the code, and it contains conduct at work and outside. \n \nVision and Mission \n \n Vision'

In [16]:
##generate embding
embedding=embedding_manager.generate_embedding(texts)
##store into vector db
vectorstore.add_document(chunks,embedding=embedding)

generateing embedding for 958 texts


Batches: 100%|██████████| 30/30 [00:01<00:00, 22.70it/s]


embeding generated with shape (958, 384)
adding 958 to vector database
succefully added 958 documents to vector store
total documents in collection 990


In [17]:
class RAGRetrieval:
    def __init__(self, vectorstore: VectorStore, embeddingmanager: EmbeddingManager):
        self.vector_store = vectorstore
        self.embeddingmanager = embeddingmanager

    def retrieve(self, query: str, k: int = 3, score_threshold: float = 0.0):
        """Retrieve the top-k matching chunks for a query using cosine similarity."""
        qembeds = self.embeddingmanager.generate_embedding([query])
        _, documents, embeddings = self.vector_store.get_all_documents()
        similarity_scores = cosine_similarity(qembeds, embeddings)[0]
        top_indices = np.argsort(similarity_scores)[::-1][:k]

        return [
            {
                "id": self.vector_store.doc_ids[i],
                "document": documents[i],
                "score": float(similarity_scores[i])
            }
            for i in top_indices
            if similarity_scores[i] >= score_threshold
        ]
rag=RAGRetrieval(vectorstore,embedding_manager)
relateddoc=rag.retrieve("what is the attention in rag")
relateddoc

generateing embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.82it/s]

embeding generated with shape (1, 384)


[{'id': 'doc_aeca070c_299',
  'document': 'Institutions, 2009, ragging has been defined as:',
  'score': 0.5654853306397851},
 {'id': 'doc_45585a3e_294',
  'document': 'ragging. \nIt is brought to the notice of all stakeholders that ragging is a criminal offence',
  'score': 0.5053980947117571},
 {'id': 'doc_e3acfb50_440',
  'document': '7. To assist the College anti-ragging Committee in preventing ragging in the',
  'score': 0.49426831774487634}]

In [28]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if not api_key:
    print("Critical Error: GOOGLE_API_KEY or GEMINI_API_KEY is missing from environment.")


In [29]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(
    model="gemini-3.5-flash",
    model_provider="google_genai",
    api_key=api_key,
    temperature=0.1,
)
def simple_rag(query, retriever, llm, top_k=3):
    ## retrieve the context
    result = retriever.retrieve(query, top_k)
    context = "\n\n".join(doc["document"] for doc in result)
    if not context:
        return "no related context found"
    prompt = f"""
use the following context and answer the question concisely
context:{context}
question:{query}
Answer:"""
    response = llm.invoke(prompt)
    return response.content


In [32]:
anwer=simple_rag("what the document is about",rag,llm)
print(anwer)

generateing embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.03it/s]

embeding generated with shape (1, 384)


[{'type': 'text', 'text': 'Based on the context, the document is a **Policy Document for St. Paul’s College, Kalamassery**, which outlines guidelines and regulations regarding extension activities, teaching staff responsibilities, and student conduct.', 'extras': {'signature': 'EqQLCqELARFNMg8YguWQb+s0phKAXj4AUxcbqIYDv5NJ8Z/a5T78BLo8mhPfR245+SJCMNvfyEjHQ+hffyD7qVw/18tkjiZSzdvJFT4knpEktgYhxAt/betJgujsDS2Z47/8jGHCW7kU0D7aonX9hDOZFiThUYoUTmLGY0F7TQFs795xLuCGP4gja4peitX1IOH7+yNh7xlAX+MzEZGLfK1Vz6u4p7A5bIgagmEhwyJDjlI/CbhveQIy+xvkoRfiP5VI5fARqXaO7saLiX9C2Jpvpj8PWjOJD1cDYplYArTdTnSjItuMtRcEyaPE7WTgLqAldjJcv7uA7s02UrqOHW/R013jEY4GiS27UKeUlX200xIz8vnT8RqM7wXcm8DckWZRcMgNV92xWjqrNIYXvjYLSBcQRyan6ZwEp+v72SYNbJpGt6Fbf+hLoCyzz5BbhpCe4eDO7asVBjBv3LycDCqo/eruOiTA906TNNh3irLE2v1iyGwV+4Z2GYID/NcaryMi60pz7KJUaWDfPH7FXTgJmp5TfDR88kYT2/7SHezveDLYrW+rGZA0JdfVD+LJLoWB7glq2uA53/BCbHBpozrKTLMr1LrZmYbJs5r8ontlKOzHkIqglYmPzYoJ7BMq/LE//z4fi2r9Wta7UtySROMGvqlsnK7YPsD+qX8v++VkcRn1Oj5SFzB9lv/xJqqqtnGr4uOezWjOOF8g4